<a href="https://colab.research.google.com/github/Habibaaboalhassan66/swimming-detection/blob/main/posestimation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Only butterfly**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
"""
File 2 (Butterfly Only): pose_estimation_butterfly_only.py
GP26 - Swimmer Injury Pattern Detection System
Habiba Tarek, German University in Cairo

Same as pose_estimation.py but runs ONLY on butterfly videos
using the newly retrained model: swimmer-butterfly-front1/5
"""

import os
import json
import base64
import requests
import numpy as np
import cv2
from pathlib import Path
from google.colab import drive

# ─── Mount Drive ───────────────────────────────────────────────────────────────
drive.mount('/content/drive', force_remount=False)

# ─── Paths ─────────────────────────────────────────────────────────────────────
STROKE_RESULTS_PATH = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/stroke_detection_results.json"
POSE_RESULTS_PATH   = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/pose_estimation_results_butterfly.json"

BUTTERFLY_TEST_DIR  = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4/butterfly"

# ─── Roboflow config ───────────────────────────────────────────────────────────
API_KEY          = "2gU1ewQS0rfbADpO4tCb"
BUTTERFLY_MODEL  = "swimmer-butterfly-front1/5"   # newly retrained model

# ─── Settings ─────────────────────────────────────────────────────────────────
SAMPLE_SIZE        = 300
KEYPOINT_THRESHOLD = 0.70
KEYPOINT_NAMES     = ["head", "left_shoulder", "right_shoulder",
                      "left_elbow", "right_elbow", "left_wrist", "right_wrist"]

SKELETON_CONNECTIONS = [
    (0, 1), (0, 2),
    (1, 2),
    (1, 3), (3, 5),
    (2, 4), (4, 6),
]


def load_stroke_results():
    """Load File 1 output from Drive."""
    if not os.path.exists(STROKE_RESULTS_PATH):
        raise FileNotFoundError(
            f"Stroke results not found at {STROKE_RESULTS_PATH}. "
            "Run File 1 first."
        )
    with open(STROKE_RESULTS_PATH, 'r') as f:
        return json.load(f)


def load_existing_pose_results():
    """Load existing pose results so we can merge butterfly results into them."""
    if os.path.exists(POSE_RESULTS_PATH):
        with open(POSE_RESULTS_PATH, 'r') as f:
            return json.load(f)
    return {}


def select_frames(all_frames: list) -> list:
    """Select middle 80% of frames, up to SAMPLE_SIZE evenly spaced."""
    n = len(all_frames)
    start = int(n * 0.10)
    end   = int(n * 0.90)
    subset = all_frames[start:end]
    if len(subset) <= SAMPLE_SIZE:
        return subset
    indices = np.linspace(0, len(subset) - 1, SAMPLE_SIZE, dtype=int)
    return [subset[i] for i in indices]


def encode_image_b64(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def call_roboflow_keypoint(image_b64: str) -> dict | None:
    url = f"https://detect.roboflow.com/{BUTTERFLY_MODEL}?api_key={API_KEY}"
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
    try:
        resp = requests.post(url, data=image_b64, headers=headers, timeout=15)
        resp.raise_for_status()
        return resp.json()
    except Exception as e:
        print(f"    [API ERROR] {e}")
        return None


def parse_keypoints(api_response: dict) -> dict:
    """Extract keypoints from Roboflow response. Trust confidence only — no geometric overrides."""
    keypoints = {}
    predictions = api_response.get("predictions", [])
    if not predictions:
        return {kp: {"x": None, "y": None, "confidence": 0.0, "visible": False}
                for kp in KEYPOINT_NAMES}

    best = max(predictions, key=lambda p: p.get("confidence", 0))
    raw_kps = {kp.get("class", "").lower(): kp
               for kp in best.get("keypoints", [])}

    for kp_name in KEYPOINT_NAMES:
        if kp_name in raw_kps:
            kp   = raw_kps[kp_name]
            conf = float(kp.get("confidence", 0.0))
            x    = float(kp.get("x", 0))
            y    = float(kp.get("y", 0))
            keypoints[kp_name] = {
                "x": x, "y": y,
                "confidence": conf,
                "visible": conf >= KEYPOINT_THRESHOLD
            }
        else:
            keypoints[kp_name] = {
                "x": None, "y": None,
                "confidence": 0.0,
                "visible": False
            }
    return keypoints


def compute_measurements(keypoints: dict) -> dict:
    ls = keypoints.get("left_shoulder")
    rs = keypoints.get("right_shoulder")
    hd = keypoints.get("head")

    measurements = {
        "shoulder_asymmetry": None,
        "shoulder_width": None,
        "head_offset": None,
        "valid": False
    }

    if ls["visible"] and rs["visible"] and ls["x"] is not None:
        measurements["shoulder_asymmetry"] = abs(ls["y"] - rs["y"])
        measurements["shoulder_width"]     = abs(ls["x"] - rs["x"])
        measurements["valid"] = True
        if hd["visible"] and hd["x"] is not None:
            mid_x = (ls["x"] + rs["x"]) / 2
            measurements["head_offset"] = abs(hd["x"] - mid_x)

    return measurements


def process_video(video_name: str) -> dict:
    print(f"\n  Processing: {video_name} (butterfly)")

    video_folder = Path(BUTTERFLY_TEST_DIR) / video_name
    if not video_folder.exists():
        print(f"    [WARN] Folder not found: {video_folder}")
        return {"error": f"folder not found: {video_folder}", "stroke": "butterfly"}

    # Sort frames by name to ensure correct temporal order (frame_000000 → end)
    all_files = sorted(video_folder.glob("*.jpg"))
    if not all_files:
        all_files = sorted(video_folder.glob("*.png"))

    if not all_files:
        print(f"    [WARN] No frames found in {video_folder}")
        return {"error": "no frames found", "stroke": "butterfly"}

    selected = select_frames(list(all_files))
    print(f"    Frames: {len(all_files)} total → {len(selected)} selected")

    frame_results = []
    api_errors    = 0

    for frame_path in selected:
        img_bgr = cv2.imread(str(frame_path))
        if img_bgr is None:
            continue

        b64      = encode_image_b64(str(frame_path))
        response = call_roboflow_keypoint(b64)

        if response is None:
            api_errors += 1
            continue

        keypoints    = parse_keypoints(response)
        measurements = compute_measurements(keypoints)

        frame_results.append({
            "frame": frame_path.name,
            "keypoints": {
                kp: {
                    "x": keypoints[kp]["x"],
                    "y": keypoints[kp]["y"],
                    "confidence": keypoints[kp]["confidence"],
                    "visible": keypoints[kp]["visible"]
                }
                for kp in KEYPOINT_NAMES
            },
            "measurements": measurements
        })

    if not frame_results:
        return {"error": "all API calls failed", "stroke": "butterfly",
                "api_errors": api_errors}

    valid_frames = [f for f in frame_results if f["measurements"]["valid"]]
    asym_vals    = [f["measurements"]["shoulder_asymmetry"] for f in valid_frames
                    if f["measurements"]["shoulder_asymmetry"] is not None]
    width_vals   = [f["measurements"]["shoulder_width"] for f in valid_frames
                    if f["measurements"]["shoulder_width"] is not None]
    offset_vals  = [f["measurements"]["head_offset"] for f in valid_frames
                    if f["measurements"]["head_offset"] is not None]

    summary = {
        "mean_shoulder_asymmetry": float(np.mean(asym_vals)) if asym_vals else None,
        "mean_shoulder_width":     float(np.mean(width_vals)) if width_vals else None,
        "mean_head_offset":        float(np.mean(offset_vals)) if offset_vals else None,
        "frames_processed":        len(frame_results),
        "frames_valid":            len(valid_frames),
        "api_errors":              api_errors
    }

    print(f"    Valid frames: {len(valid_frames)}/{len(frame_results)}")
    if asym_vals:
        print(f"    Mean shoulder asymmetry: {summary['mean_shoulder_asymmetry']:.2f}px")
    if width_vals:
        print(f"    Mean shoulder width:     {summary['mean_shoulder_width']:.2f}px")

    return {"stroke": "butterfly", "frames": frame_results, "summary": summary}


def run_pose_estimation_butterfly():
    print("=" * 60)
    print("FILE 2 (Butterfly Only): Pose Estimation")
    print(f"Model: {BUTTERFLY_MODEL}")
    print("=" * 60)

    # Load File 1 results to find butterfly videos
    stroke_results = load_stroke_results()
    butterfly_videos = [
        name for name, res in stroke_results.items()
        if res.get("detected_stroke") == "butterfly"
    ]
    print(f"\n  Found {len(butterfly_videos)} butterfly videos: {butterfly_videos}")

    # Load existing pose results to preserve breaststroke results
    all_pose_results = load_existing_pose_results()
    print(f"  Loaded existing pose results ({len(all_pose_results)} videos) — breaststroke kept as-is")

    # Process only butterfly videos
    for video_name in butterfly_videos:
        result = process_video(video_name)
        all_pose_results[video_name] = result  # overwrite butterfly only

    # Save merged results
    os.makedirs(os.path.dirname(POSE_RESULTS_PATH), exist_ok=True)
    with open(POSE_RESULTS_PATH, 'w') as f:
        json.dump(all_pose_results, f, indent=2, default=str)
    print(f"\n✅ Results saved → {POSE_RESULTS_PATH}")

    print("\n── Summary ──────────────────────────────────────────")
    for vname, res in all_pose_results.items():
        if "error" in res:
            print(f"  {vname}: ERROR — {res['error']}")
        else:
            s = res.get("summary", {})
            print(f"  {vname} ({res['stroke']}): "
                  f"{s.get('frames_valid', 0)}/{s.get('frames_processed', 0)} valid frames")
    print("=" * 60)

    return all_pose_results


if __name__ == "__main__":
    run_pose_estimation_butterfly()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FILE 2 (Butterfly Only): Pose Estimation
Model: swimmer-butterfly-front1/5

  Found 3 butterfly videos: ['butterfly_front_S07', 'butterfly_front_S08', 'butterfly_front_S09']
  Loaded existing pose results (3 videos) — breaststroke kept as-is

  Processing: butterfly_front_S07 (butterfly)
    Frames: 203 total → 162 selected
    Valid frames: 57/162
    Mean shoulder asymmetry: 1.44px
    Mean shoulder width:     55.89px

  Processing: butterfly_front_S08 (butterfly)
    Frames: 224 total → 179 selected
    Valid frames: 76/179
    Mean shoulder asymmetry: 0.95px
    Mean shoulder width:     39.58px

  Processing: butterfly_front_S09 (butterfly)
    Frames: 171 total → 136 selected
    Valid frames: 54/136
    Mean shoulder asymmetry: 1.17px
    Mean shoulder width:     38.81px

✅ Results saved → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/po

**final pose**

In [ ]:
"""
File 2c: visualization_butterfly.py
GP26 - Swimmer Injury Pattern Detection System
Habiba Tarek, German University in Cairo

Butterfly-only visualization script.
Reads pose_estimation_results.json, filters low confidence frames,
draws green skeleton overlay and saves to:
  Models/visualizations/butterfly/test/{video_name}/{frame_name}.jpg

Run AFTER pose_estimation_butterfly_only.py has completed.
"""

import os
import json
import cv2
import numpy as np
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

# ─── Paths ─────────────────────────────────────────────────────────────────────
POSE_RESULTS_PATH  = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/pose_estimation_results_butterfly.json"
BUTTERFLY_TEST_DIR = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4/butterfly"
VIZ_DIR            = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/visualizations/butterfly/test"

# ─── Keypoint config ───────────────────────────────────────────────────────────
KEYPOINT_NAMES = ["head", "left_shoulder", "right_shoulder",
                  "left_elbow", "right_elbow", "left_wrist", "right_wrist"]

SKELETON_CONNECTIONS = [
    (0, 1), (0, 2),   # head → shoulders
    (1, 2),           # left_shoulder → right_shoulder
    (1, 3), (3, 5),   # left arm chain
    (2, 4), (4, 6),   # right arm chain
]

def compute_confidence_threshold(pose_results: dict, stroke: str = "butterfly") -> float:
    """
    Automatically compute confidence threshold from the data.
    Collects all non-zero average keypoint confidences, then sets the threshold
    at mean - 1 standard deviation to keep only clearly high-confidence frames.
    """
    all_confs = []
    for video_name, result in pose_results.items():
        if result.get("stroke") != stroke or "error" in result:
            continue
        for frame_data in result.get("frames", []):
            keypoints = frame_data.get("keypoints", {})
            kp_confs  = [
                kp["confidence"] for kp in keypoints.values()
                if kp.get("confidence") is not None
            ]
            if kp_confs:
                avg = sum(kp_confs) / len(kp_confs)
                if avg > 0.0:  # ignore completely empty frames
                    all_confs.append(avg)

    if not all_confs:
        return 0.75  # fallback

    arr    = np.array(all_confs)
    thresh = float(np.mean(arr) - np.std(arr))
    thresh = round(max(thresh, 0.50), 2)  # never go below 0.50

    print(f"  Non-zero frames: {len(arr)} | mean={np.mean(arr):.3f} | "
          f"std={np.std(arr):.3f} | threshold={thresh:.2f}")
    print(f"  Auto confidence threshold: {thresh:.2f} "
          f"(from {len(arr)} non-zero frames across all {stroke} videos)")
    return thresh


# ══════════════════════════════════════════════════════════════════════════════
# FILTER
# ══════════════════════════════════════════════════════════════════════════════

def filter_low_confidence_frames(frames: list, threshold: float) -> list:
    """
    Filter out frames where average keypoint confidence < threshold.
    Returns only frames with avg keypoint confidence >= threshold.
    """
    filtered = []
    for frame_data in frames:
        keypoints = frame_data.get("keypoints", {})
        kp_confs  = [
            kp["confidence"] for kp in keypoints.values()
            if kp.get("confidence") is not None
        ]
        if not kp_confs:
            continue
        avg_conf = sum(kp_confs) / len(kp_confs)
        if avg_conf >= threshold:
            filtered.append(frame_data)
    return filtered


# ══════════════════════════════════════════════════════════════════════════════
# VISUALIZATION
# ══════════════════════════════════════════════════════════════════════════════

def draw_skeleton(image: np.ndarray, keypoints: dict) -> np.ndarray:
    """Solid green dots scaled by shoulder width, with thick green skeleton lines."""
    img = image.copy()

    GREEN_BRIGHT = (0, 255, 0)
    GREEN_DARK   = (0, 180, 0)
    LINE_GREEN   = (0, 220, 0)
    OCCLUDED     = (80, 80, 80)

    kp_coords = {}
    for kp_name in KEYPOINT_NAMES:
        kp = keypoints.get(kp_name, {})
        if kp.get("x") is None:
            continue
        x, y = int(kp["x"]), int(kp["y"])
        kp_coords[kp_name] = (x, y, kp.get("visible", False))

    # Fixed small dot radius — no dynamic scaling
    dot_radius = 3

    # Lines first
    for i, j in SKELETON_CONNECTIONS:
        n1, n2 = KEYPOINT_NAMES[i], KEYPOINT_NAMES[j]
        if n1 in kp_coords and n2 in kp_coords:
            p1 = kp_coords[n1][:2]
            p2 = kp_coords[n2][:2]
            if kp_coords[n1][2] and kp_coords[n2][2]:
                cv2.line(img, p1, p2, LINE_GREEN, 1, cv2.LINE_AA)

    # Dots on top
    for kp_name, (x, y, visible) in kp_coords.items():
        if visible:
            cv2.circle(img, (x, y), dot_radius, GREEN_DARK,   -1, cv2.LINE_AA)
            cv2.circle(img, (x, y), 2,          GREEN_BRIGHT, -1, cv2.LINE_AA)
        else:
            cv2.circle(img, (x, y), 2, OCCLUDED, -1, cv2.LINE_AA)

    return img


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

def run_visualization_butterfly():
    print("=" * 60)
    print("FILE 2c: Butterfly Visualization")
    print(f"Output: {VIZ_DIR}/{{video_name}}/{{frame_name}}.jpg")
    print("=" * 60)

    if not os.path.exists(POSE_RESULTS_PATH):
        raise FileNotFoundError(
            f"Pose results not found at {POSE_RESULTS_PATH}. "
            "Run pose_estimation_butterfly_only.py first."
        )

    with open(POSE_RESULTS_PATH, 'r') as f:
        pose_results = json.load(f)

    # ── Step 1: Auto-compute confidence threshold from all butterfly frames ────
    print("\nComputing confidence threshold automatically...")
    threshold = compute_confidence_threshold(pose_results, stroke="butterfly")
    print(f"  Using threshold: {threshold:.2f}")

    total_saved = 0

    # ── Step 2: Filter and visualize ──────────────────────────────────────────
    for video_name, result in pose_results.items():
        if result.get("stroke") != "butterfly":
            continue
        if "error" in result:
            print(f"\n  [SKIP] {video_name} — {result['error']}")
            continue

        frames = result.get("frames", [])

        # Sort by frame name for correct temporal order
        frames_sorted = sorted(frames, key=lambda f: f.get("frame", ""))

        # Filter out low confidence frames using auto threshold
        frames_filtered = filter_low_confidence_frames(frames_sorted, threshold)

        print(f"\n  {video_name}: {len(frames_sorted)} total → "
              f"{len(frames_filtered)} after confidence filter (>= {threshold:.2f})")

        viz_video_dir = os.path.join(VIZ_DIR, video_name)
        os.makedirs(viz_video_dir, exist_ok=True)

        saved = 0
        for frame_data in frames_filtered:
            frame_name = frame_data.get("frame")
            keypoints  = frame_data.get("keypoints", {})

            # Skip frames where both shoulders not visible
            ls = keypoints.get("left_shoulder", {})
            rs = keypoints.get("right_shoulder", {})
            if not (ls.get("visible") and rs.get("visible")):
                continue

            frame_path = os.path.join(BUTTERFLY_TEST_DIR, video_name, frame_name)
            if not os.path.exists(frame_path):
                print(f"    [WARN] Frame not found: {frame_path}")
                continue

            img = cv2.imread(frame_path)
            if img is None:
                continue

            viz       = draw_skeleton(img, keypoints)
            save_path = os.path.join(viz_video_dir, frame_name)
            cv2.imwrite(save_path, viz)
            saved += 1

        print(f"    Saved {saved} visualizations → {viz_video_dir}")
        total_saved += saved

    print(f"\n✅ Total butterfly visualizations saved: {total_saved}")
    print(f"   Location: {VIZ_DIR}")
    print("=" * 60)


if __name__ == "__main__":
    run_visualization_butterfly()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FILE 2c: Butterfly Visualization
Output: /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/visualizations/butterfly/test/{video_name}/{frame_name}.jpg

Computing confidence threshold automatically...
  Non-zero frames: 204 | mean=0.884 | std=0.168 | threshold=0.72
  Auto confidence threshold: 0.72 (from 204 non-zero frames across all butterfly videos)
  Using threshold: 0.72

  butterfly_front_S07: 162 total → 52 after confidence filter (>= 0.72)
    Saved 52 visualizations → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/visualizations/butterfly/test/butterfly_front_S07

  butterfly_front_S08: 179 total → 76 after confidence filter (>= 0.72)
    Saved 76 visualizations → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/visualizations/butterfly/test/butterfly_front_S08

  butterfly_front_S09: 136 total → 54 after confid

**Breaststroke**

In [ ]:
"""
File 2d: visualization_breaststroke.py
GP26 - Swimmer Injury Pattern Detection System
Habiba Tarek, German University in Cairo

Breaststroke-only visualization script.
Reads pose_estimation_results.json, auto-computes confidence threshold,
filters low confidence and overlapping frames, draws green skeleton overlay
and saves to:
  Models/visualizations/breaststroke/test/{video_name}/{frame_name}.jpg

Run AFTER pose_estimation.py has completed.
"""

import os
import json
import cv2
import numpy as np
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

# ─── Paths ─────────────────────────────────────────────────────────────────────
breast_path =       "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/pose_estimation_results_breaststroke.json"
BREASTSTROKE_TEST_DIR = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4/breaststroke_testing"
VIZ_DIR               = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/visualizations/breaststroke/test"

# ─── Keypoint config ───────────────────────────────────────────────────────────
KEYPOINT_NAMES = ["head", "left_shoulder", "right_shoulder",
                  "left_elbow", "right_elbow", "left_wrist", "right_wrist"]

SKELETON_CONNECTIONS = [
    (0, 1), (0, 2),   # head → shoulders
    (1, 2),           # left_shoulder → right_shoulder
    (1, 3), (3, 5),   # left arm chain
    (2, 4), (4, 6),   # right arm chain
]

MIN_DISTANCE = 10  # px — keypoints closer than this = overlapping


# ══════════════════════════════════════════════════════════════════════════════
# THRESHOLD
# ══════════════════════════════════════════════════════════════════════════════

def compute_confidence_threshold(pose_results: dict, stroke: str = "breaststroke") -> float:
    """
    Auto-compute confidence threshold from non-zero frames using mean - std.
    """
    all_confs = []
    for video_name, result in pose_results.items():
        if result.get("stroke") != stroke or "error" in result:
            continue
        for frame_data in result.get("frames", []):
            keypoints = frame_data.get("keypoints", {})
            kp_confs  = [
                kp["confidence"] for kp in keypoints.values()
                if kp.get("confidence") is not None
            ]
            if kp_confs:
                avg = sum(kp_confs) / len(kp_confs)
                if avg > 0.0:
                    all_confs.append(avg)

    if not all_confs:
        return 0.75

    arr    = np.array(all_confs)
    thresh = float(np.mean(arr) - 2 * np.std(arr))
    thresh = round(max(thresh, 0.50), 2)

    print(f"  Non-zero frames: {len(arr)} | mean={np.mean(arr):.3f} | "
          f"std={np.std(arr):.3f} | threshold={thresh:.2f}")
    print(f"  Auto confidence threshold: {thresh:.2f} "
          f"(from {len(arr)} non-zero frames across all {stroke} videos)")
    return thresh


# ══════════════════════════════════════════════════════════════════════════════
# FILTERS
# ══════════════════════════════════════════════════════════════════════════════

def filter_low_confidence_frames(frames: list, threshold: float) -> list:
    """Filter out frames where average keypoint confidence < threshold."""
    filtered = []
    for frame_data in frames:
        keypoints = frame_data.get("keypoints", {})
        kp_confs  = [
            kp["confidence"] for kp in keypoints.values()
            if kp.get("confidence") is not None
        ]
        if not kp_confs:
            continue
        if (sum(kp_confs) / len(kp_confs)) >= threshold:
            filtered.append(frame_data)
    return filtered


def filter_overlapping_keypoints(frames: list, min_distance: int = MIN_DISTANCE) -> list:
    """Filter out frames where any two visible keypoints are within min_distance pixels."""
    filtered = []
    for frame_data in frames:
        keypoints = frame_data.get("keypoints", {})
        positions = []
        for kp_name in KEYPOINT_NAMES:
            kp = keypoints.get(kp_name, {})
            if kp.get("visible") and kp.get("x") is not None:
                positions.append((kp["x"], kp["y"]))

        overlapping = False
        for i in range(len(positions)):
            for j in range(i + 1, len(positions)):
                dx   = positions[i][0] - positions[j][0]
                dy   = positions[i][1] - positions[j][1]
                dist = (dx**2 + dy**2) ** 0.5
                if dist < min_distance:
                    overlapping = True
                    break
            if overlapping:
                break

        if not overlapping:
            filtered.append(frame_data)
    return filtered


# ══════════════════════════════════════════════════════════════════════════════
# VISUALIZATION
# ══════════════════════════════════════════════════════════════════════════════

def draw_skeleton(image: np.ndarray, keypoints: dict) -> np.ndarray:
    """Small green dots (3px) with thin green lines (1px)."""
    img = image.copy()

    GREEN_BRIGHT = (0, 255, 0)
    GREEN_DARK   = (0, 180, 0)
    LINE_GREEN   = (0, 220, 0)
    OCCLUDED     = (80, 80, 80)

    kp_coords = {}
    for kp_name in KEYPOINT_NAMES:
        kp = keypoints.get(kp_name, {})
        if kp.get("x") is None:
            continue
        x, y = int(kp["x"]), int(kp["y"])
        kp_coords[kp_name] = (x, y, kp.get("visible", False))

    # Lines first
    for i, j in SKELETON_CONNECTIONS:
        n1, n2 = KEYPOINT_NAMES[i], KEYPOINT_NAMES[j]
        if n1 in kp_coords and n2 in kp_coords:
            p1 = kp_coords[n1][:2]
            p2 = kp_coords[n2][:2]
            if kp_coords[n1][2] and kp_coords[n2][2]:
                cv2.line(img, p1, p2, LINE_GREEN, 1, cv2.LINE_AA)

    # Dots on top
    for kp_name, (x, y, visible) in kp_coords.items():
        if visible:
            cv2.circle(img, (x, y), 3, GREEN_DARK,   -1, cv2.LINE_AA)
            cv2.circle(img, (x, y), 2, GREEN_BRIGHT, -1, cv2.LINE_AA)
        else:
            cv2.circle(img, (x, y), 2, OCCLUDED, -1, cv2.LINE_AA)

    return img


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

def run_visualization_breaststroke():
    print("=" * 60)
    print("FILE 2d: Breaststroke Visualization")
    print(f"Output: {VIZ_DIR}/{{video_name}}/{{frame_name}}.jpg")
    print("=" * 60)

    POSE_RESULTS_PATH = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/pose_estimation_results_breaststroke.json"

    if not os.path.exists(POSE_RESULTS_PATH):
        raise FileNotFoundError(
            f"Pose results not found at {POSE_RESULTS_PATH}. "
            "Run pose_estimation.py first."
        )

    with open(POSE_RESULTS_PATH, 'r') as f:
        pose_results = json.load(f)

    # Step 1 — Auto-compute confidence threshold
    print("\nComputing confidence threshold automatically...")
    threshold = compute_confidence_threshold(pose_results, stroke="breaststroke")
    print(f"  Using threshold: {threshold:.2f}")

    total_saved = 0

    for video_name, result in pose_results.items():
        if result.get("stroke") != "breaststroke":
            continue
        if "error" in result:
            print(f"\n  [SKIP] {video_name} — {result['error']}")
            continue

        frames = result.get("frames", [])
        frames_sorted   = sorted(frames, key=lambda f: f.get("frame", ""))

        # Step 2 — Filter low confidence frames
        frames_filtered = filter_low_confidence_frames(frames_sorted, threshold)

        # Step 3 — Filter overlapping keypoints
        frames_filtered = filter_overlapping_keypoints(frames_filtered)

        print(f"\n  {video_name}: {len(frames_sorted)} total → "
              f"{len(frames_filtered)} after filtering (>= {threshold:.2f})")

        viz_video_dir = os.path.join(VIZ_DIR, video_name)
        os.makedirs(viz_video_dir, exist_ok=True)

        saved = 0
        for frame_data in frames_filtered:
            frame_name = frame_data.get("frame")
            keypoints  = frame_data.get("keypoints", {})

            ls = keypoints.get("left_shoulder", {})
            rs = keypoints.get("right_shoulder", {})
            if not (ls.get("visible") and rs.get("visible")):
                continue

            frame_path = os.path.join(BREASTSTROKE_TEST_DIR, video_name, frame_name)
            if not os.path.exists(frame_path):
                print(f"    [WARN] Frame not found: {frame_path}")
                continue

            img = cv2.imread(frame_path)
            if img is None:
                continue

            viz       = draw_skeleton(img, keypoints)
            save_path = os.path.join(viz_video_dir, frame_name)
            cv2.imwrite(save_path, viz)
            saved += 1

        print(f"    Saved {saved} visualizations → {viz_video_dir}")
        total_saved += saved

    print(f"\n✅ Total breaststroke visualizations saved: {total_saved}")
    print(f"   Location: {VIZ_DIR}")
    print("=" * 60)


if __name__ == "__main__":
    run_visualization_breaststroke()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FILE 2d: Breaststroke Visualization
Output: /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/visualizations/breaststroke/test/{video_name}/{frame_name}.jpg


FileNotFoundError: Pose results not found at /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/pose_estimation_results_breaststroke.json. Run pose_estimation.py first.

In [ ]:
# @title
"""
File 2: pose_estimation_breaststroke.py
GP26 - Swimmer Injury Pattern Detection System
Habiba Tarek, German University in Cairo

Extracts 7 upper-body keypoints per frame using fine-tuned Roboflow
breaststroke model. Reads stroke detection results from Drive.
Saves breaststroke-only pose results to Drive.

FIXES FROM PREVIOUS VERSION:
- Switched to WORKFLOW endpoint (direct model endpoint returns empty predictions)
- Fixed frame selection to use middle 80% correctly
- Added retry logic (3 attempts, 30s timeout)
- Saves to pose_estimation_results_breaststroke.json (separate from butterfly)
- Saves frame_path for Phase 1 visualization
"""

import os
import json
import base64
import requests
import numpy as np
import cv2
import math
import time
from pathlib import Path
from google.colab import drive

# ─── Mount Drive ───────────────────────────────────────────────────────────────
drive.mount('/content/drive', force_remount=False)

# ─── Paths ─────────────────────────────────────────────────────────────────────
STROKE_RESULTS_PATH   = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/stroke_detection_results.json"
POSE_RESULTS_PATH     = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/pose_estimation_results_breaststroke.json"
BREASTSTROKE_TEST_DIR = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4/breaststroke_testing"

# ─── Roboflow config ───────────────────────────────────────────────────────────
API_KEY   = "2gU1ewQS0rfbADpO4tCb"
WORKSPACE = "habibas-workspace-jcdgt"
WORKFLOW  = "breaststroke-swimmer-baseline-1780255913826"

# ─── Settings ──────────────────────────────────────────────────────────────────
SAMPLE_SIZE        = 300
KEYPOINT_THRESHOLD = 0.70
KEYPOINT_NAMES     = ["head", "left_shoulder", "right_shoulder",
                      "left_elbow", "right_elbow", "left_wrist", "right_wrist"]

SKELETON_CONNECTIONS = [
    (0, 1), (0, 2),
    (1, 2),
    (1, 3), (3, 5),
    (2, 4), (4, 6),
]

# ══════════════════════════════════════════════════════════════════════════════
# LOAD STROKE RESULTS
# ══════════════════════════════════════════════════════════════════════════════
def load_stroke_results():
    if not os.path.exists(STROKE_RESULTS_PATH):
        raise FileNotFoundError(
            f"Stroke results not found at {STROKE_RESULTS_PATH}. "
            "Run File 1 first."
        )
    with open(STROKE_RESULTS_PATH, 'r') as f:
        return json.load(f)


# ══════════════════════════════════════════════════════════════════════════════
# FRAME SELECTION
# ══════════════════════════════════════════════════════════════════════════════
def select_frames(all_frames: list) -> list:
    """
    Breaststroke: middle 80% of frames.
    Skip first 10% (entry) and last 10% (finish).
    Sample up to SAMPLE_SIZE evenly.
    """
    n     = len(all_frames)
    start = int(n * 0.10)
    end   = int(n * 0.90)
    subset = all_frames[start:end]

    if len(subset) <= SAMPLE_SIZE:
        return subset

    indices = np.linspace(0, len(subset) - 1, SAMPLE_SIZE, dtype=int)
    return [subset[i] for i in indices]


# ══════════════════════════════════════════════════════════════════════════════
# API
# ══════════════════════════════════════════════════════════════════════════════
def encode_image_b64(image_path: str) -> str:
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def call_roboflow_keypoint(image_b64: str) -> dict | None:
    """
    Call Roboflow WORKFLOW endpoint — confirmed working.
    Direct model endpoint returns empty predictions.
    Retries 3 times with 30s timeout.
    """
    url = f"https://detect.roboflow.com/infer/workflows/{WORKSPACE}/{WORKFLOW}"

    for attempt in range(3):
        try:
            resp = requests.post(
                url,
                json={
                    "api_key": API_KEY,
                    "inputs": {
                        "image": {
                            "type":  "base64",
                            "value": image_b64
                        }
                    }
                },
                timeout=30
            )
            resp.raise_for_status()
            data    = resp.json()
            outputs = data.get("outputs", [])
            if not outputs:
                return None
            predictions = outputs[0].get("predictions", {}).get("predictions", [])
            if not predictions:
                return None
            best = max(predictions, key=lambda p: p.get("confidence", 0))
            return {"predictions": [best]}

        except Exception as e:
            print(f"    [API ERROR attempt {attempt+1}/3] {e}")
            if attempt < 2:
                time.sleep(2)

    return None


# ══════════════════════════════════════════════════════════════════════════════
# KEYPOINT PARSING
# ══════════════════════════════════════════════════════════════════════════════
def parse_keypoints(api_response: dict) -> dict:
    """
    Extract 7 keypoints from Roboflow response.
    Applies geometric validation and wrist mirroring.
    """
    keypoints   = {}
    predictions = api_response.get("predictions", [])

    if not predictions:
        return {kp: {"x": None, "y": None, "confidence": 0.0, "visible": False}
                for kp in KEYPOINT_NAMES}

    best    = max(predictions, key=lambda p: p.get("confidence", 0))
    raw_kps = {kp.get("class", "").lower(): kp
               for kp in best.get("keypoints", [])}

    for kp_name in KEYPOINT_NAMES:
        if kp_name in raw_kps:
            kp   = raw_kps[kp_name]
            conf = float(kp.get("confidence", 0.0))
            x    = float(kp.get("x", 0))
            y    = float(kp.get("y", 0))
            keypoints[kp_name] = {
                "x": x, "y": y,
                "confidence": conf,
                "visible":    conf >= KEYPOINT_THRESHOLD
            }
        else:
            keypoints[kp_name] = {
                "x": None, "y": None,
                "confidence": 0.0,
                "visible":    False
            }

    # ── Geometric validation ──────────────────────────────────────────────────
    head = keypoints.get("head")
    ls   = keypoints.get("left_shoulder")
    rs   = keypoints.get("right_shoulder")

    # Head must be above shoulders
    if head["visible"] and ls["visible"] and rs["visible"]:
        shoulder_y = (ls["y"] + rs["y"]) / 2
        if head["y"] > shoulder_y:
            head["visible"] = False

    # Elbows within 1.5× shoulder width
    if ls["visible"] and rs["visible"]:
        sw = abs(ls["x"] - rs["x"])
        for side, elbow_name in [("left", "left_elbow"), ("right", "right_elbow")]:
            elbow = keypoints.get(elbow_name)
            if elbow["visible"] and sw > 0:
                ref_x = ls["x"] if side == "left" else rs["x"]
                if abs(elbow["x"] - ref_x) > 1.5 * sw:
                    elbow["visible"] = False

    # Wrists within 1.5× shoulder width of their elbow
    for side, wrist_name, elbow_name in [
        ("left",  "left_wrist",  "left_elbow"),
        ("right", "right_wrist", "right_elbow")
    ]:
        wrist = keypoints.get(wrist_name)
        elbow = keypoints.get(elbow_name)
        if (wrist["visible"] and elbow["visible"]
                and ls["visible"] and rs["visible"]):
            sw = abs(ls["x"] - rs["x"])
            if abs(wrist["x"] - elbow["x"]) > 1.5 * sw:
                wrist["visible"] = False

    # ── Wrist mirroring ───────────────────────────────────────────────────────
    lw = keypoints.get("left_wrist")
    rw = keypoints.get("right_wrist")
    if (lw["visible"] and rw["visible"]
            and lw["x"] is not None and rw["x"] is not None):
        dist = math.sqrt((lw["x"] - rw["x"])**2 + (lw["y"] - rw["y"])**2)
        if dist < 30 and ls["visible"] and rs["visible"]:
            mid_x   = (ls["x"] + rs["x"]) / 2
            rw["x"] = 2 * mid_x - lw["x"]

    return keypoints


# ══════════════════════════════════════════════════════════════════════════════
# MEASUREMENTS
# ══════════════════════════════════════════════════════════════════════════════
def compute_measurements(keypoints: dict) -> dict:
    ls = keypoints.get("left_shoulder")
    rs = keypoints.get("right_shoulder")
    hd = keypoints.get("head")

    measurements = {
        "shoulder_asymmetry": None,
        "shoulder_width":     None,
        "head_offset":        None,
        "valid":              False
    }

    if ls["visible"] and rs["visible"] and ls["x"] is not None:
        measurements["shoulder_asymmetry"] = abs(ls["y"] - rs["y"])
        measurements["shoulder_width"]     = abs(ls["x"] - rs["x"])
        measurements["valid"]              = True

        if hd["visible"] and hd["x"] is not None:
            mid_x = (ls["x"] + rs["x"]) / 2
            measurements["head_offset"] = abs(hd["x"] - mid_x)

    return measurements


# ══════════════════════════════════════════════════════════════════════════════
# PROCESS ONE VIDEO
# ══════════════════════════════════════════════════════════════════════════════
def process_video(video_name: str) -> dict:
    print(f"\n  Processing: {video_name} (breaststroke)")

    video_folder = Path(BREASTSTROKE_TEST_DIR) / video_name
    if not video_folder.exists():
        print(f"    [WARN] Folder not found: {video_folder}")
        return {"error": f"folder not found: {video_folder}",
                "stroke": "breaststroke"}

    all_files = sorted(video_folder.glob("*.jpg"))
    if not all_files:
        all_files = sorted(video_folder.glob("*.png"))

    if not all_files:
        print(f"    [WARN] No frames found in {video_folder}")
        return {"error": "no frames found", "stroke": "breaststroke"}

    selected = select_frames(list(all_files))
    print(f"    Frames: {len(all_files)} total → {len(selected)} selected")

    frame_results = []
    api_errors    = 0

    for frame_path in selected:
        img_bgr = cv2.imread(str(frame_path))
        if img_bgr is None:
            continue

        b64      = encode_image_b64(str(frame_path))
        response = call_roboflow_keypoint(b64)

        if response is None:
            api_errors += 1
            continue

        keypoints    = parse_keypoints(response)
        measurements = compute_measurements(keypoints)

        frame_results.append({
            "frame":        frame_path.name,
            "frame_path":   str(frame_path),   # saved for Phase 1 visualization
            "keypoints":    {
                kp: {
                    "x":          keypoints[kp]["x"],
                    "y":          keypoints[kp]["y"],
                    "confidence": keypoints[kp]["confidence"],
                    "visible":    keypoints[kp]["visible"]
                }
                for kp in KEYPOINT_NAMES
            },
            "measurements": measurements
        })

    if not frame_results:
        return {"error": "all API calls failed", "stroke": "breaststroke",
                "api_errors": api_errors}

    valid_frames = [f for f in frame_results if f["measurements"]["valid"]]
    asym_vals    = [f["measurements"]["shoulder_asymmetry"]
                    for f in valid_frames
                    if f["measurements"]["shoulder_asymmetry"] is not None]
    width_vals   = [f["measurements"]["shoulder_width"]
                    for f in valid_frames
                    if f["measurements"]["shoulder_width"] is not None]
    offset_vals  = [f["measurements"]["head_offset"]
                    for f in valid_frames
                    if f["measurements"]["head_offset"] is not None]

    summary = {
        "mean_shoulder_asymmetry": float(np.mean(asym_vals))   if asym_vals  else None,
        "mean_shoulder_width":     float(np.mean(width_vals))  if width_vals else None,
        "mean_head_offset":        float(np.mean(offset_vals)) if offset_vals else None,
        "frames_processed":        len(frame_results),
        "frames_valid":            len(valid_frames),
        "api_errors":              api_errors,
    }

    print(f"    Valid frames : {len(valid_frames)}/{len(frame_results)}")
    print(f"    API errors   : {api_errors}")
    if asym_vals:
        print(f"    Mean shoulder asymmetry: {summary['mean_shoulder_asymmetry']:.2f}px")
    if width_vals:
        print(f"    Mean shoulder width    : {summary['mean_shoulder_width']:.2f}px")

    return {
        "stroke":  "breaststroke",
        "frames":  frame_results,
        "summary": summary
    }


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════
def run_pose_estimation_breaststroke():
    print("=" * 60)
    print("FILE 2: Pose Estimation — Breaststroke Only")
    print(f"Model : breaststroke-swimmer-baseline (workflow)")
    print(f"Output: {POSE_RESULTS_PATH}")
    print("=" * 60)

    stroke_results = load_stroke_results()

    breaststroke_videos = [
        name for name, res in stroke_results.items()
        if res.get("detected_stroke") == "breaststroke"
    ]
    print(f"\n  Found {len(breaststroke_videos)} breaststroke videos: "
          f"{breaststroke_videos}")

    all_pose_results = {}
    for video_name in breaststroke_videos:
        result = process_video(video_name)
        all_pose_results[video_name] = result

    # Save breaststroke-only results
    os.makedirs(os.path.dirname(POSE_RESULTS_PATH), exist_ok=True)
    with open(POSE_RESULTS_PATH, 'w') as f:
        json.dump(all_pose_results, f, indent=2, default=str)
    print(f"\n✅ Results saved → {POSE_RESULTS_PATH}")

    print("\n── Summary ──────────────────────────────────────────")
    for vname, res in all_pose_results.items():
        if "error" in res:
            print(f"  {vname}: ERROR — {res['error']}")
        else:
            s = res.get("summary", {})
            print(f"  {vname} (breaststroke): "
                  f"{s.get('frames_valid', 0)}/{s.get('frames_processed', 0)} "
                  f"valid frames")
    print("=" * 60)

    return all_pose_results


run_pose_estimation_breaststroke()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FILE 2: Pose Estimation — Breaststroke Only
Model : breaststroke-swimmer-baseline (workflow)
Output: /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/pose_estimation_results_breaststroke.json

  Found 4 breaststroke videos: ['breaststroke_front_S05', 'breaststroke_front_S06', 'breaststroke_front_S07', 'breaststroke_front_S08']

  Processing: breaststroke_front_S05 (breaststroke)
    Frames: 308 total → 247 selected
    Valid frames : 28/28
    API errors   : 219
    Mean shoulder asymmetry: 0.96px
    Mean shoulder width    : 86.36px

  Processing: breaststroke_front_S06 (breaststroke)
    Frames: 234 total → 187 selected
    Valid frames : 22/22
    API errors   : 165
    Mean shoulder asymmetry: 1.36px
    Mean shoulder width    : 62.64px

  Processing: breaststroke_front_S07 (breaststroke)
    Frames: 190 total → 152 selected
    Valid frames 

{'breaststroke_front_S05': {'stroke': 'breaststroke',
  'frames': [{'frame': 'frame_000309.jpg',
    'frame_path': '/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4/breaststroke_testing/breaststroke_front_S05/frame_000309.jpg',
    'keypoints': {'head': {'x': 349.0,
      'y': 157.0,
      'confidence': 0.9840328097343445,
      'visible': True},
     'left_shoulder': {'x': 323.0,
      'y': 171.0,
      'confidence': 0.9923235177993774,
      'visible': True},
     'right_shoulder': {'x': 372.0,
      'y': 170.0,
      'confidence': 0.991811990737915,
      'visible': True},
     'left_elbow': {'x': 316.0,
      'y': 189.0,
      'confidence': 0.9807249307632446,
      'visible': True},
     'right_elbow': {'x': 381.0,
      'y': 189.0,
      'confidence': 0.9851565361022949,
      'visible': True},
     'left_wrist': {'x': 339.0,
      'y': 188.0,
      'confidence': 0.9884129166603088,
      'visible': True},
     'right_wrist': {'x': 356.0,
 

In [ ]:
# @title
"""
File 2d: visualization_breaststroke.py
GP26 - Swimmer Injury Pattern Detection System
Habiba Tarek, German University in Cairo

Breaststroke-only visualization script.
Reads pose_estimation_results_breaststroke.json,
auto-computes confidence threshold, filters low confidence
and overlapping frames, draws green skeleton overlay and saves to:
  Models/visualizations/breaststroke/test/{video_name}/{frame_name}.jpg

Run AFTER pose_estimation_breaststroke.py has completed.
"""

import os
import json
import cv2
import numpy as np
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

# ─── Paths ─────────────────────────────────────────────────────────────────────
POSE_RESULTS_PATH     = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/pose_estimation_results_breaststroke.json"
BREASTSTROKE_TEST_DIR = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Dataset/Video/processed_frames_v4/breaststroke_testing"
VIZ_DIR               = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/visualizations/breaststroke/test"

# ─── Keypoint config ───────────────────────────────────────────────────────────
KEYPOINT_NAMES = ["head", "left_shoulder", "right_shoulder",
                  "left_elbow", "right_elbow", "left_wrist", "right_wrist"]

SKELETON_CONNECTIONS = [
    (0, 1), (0, 2),
    (1, 2),
    (1, 3), (3, 5),
    (2, 4), (4, 6),
]

MIN_DISTANCE = 10  # px


# ══════════════════════════════════════════════════════════════════════════════
# THRESHOLD
# ══════════════════════════════════════════════════════════════════════════════
def compute_confidence_threshold(pose_results: dict,
                                 stroke: str = "breaststroke") -> float:
    """
    Auto-compute confidence threshold using mean - 2×std from all
    non-zero average keypoint confidences across all breaststroke frames.
    """
    all_confs = []
    for video_name, result in pose_results.items():
        if result.get("stroke") != stroke or "error" in result:
            continue
        for frame_data in result.get("frames", []):
            keypoints = frame_data.get("keypoints", {})
            kp_confs  = [
                kp["confidence"] for kp in keypoints.values()
                if kp.get("confidence") is not None
            ]
            if kp_confs:
                avg = sum(kp_confs) / len(kp_confs)
                if avg > 0.0:
                    all_confs.append(avg)

    if not all_confs:
        return 0.75

    arr    = np.array(all_confs)
    thresh = float(np.mean(arr) - 2 * np.std(arr))
    thresh = round(max(thresh, 0.50), 2)

    print(f"  Non-zero frames: {len(arr)} | mean={np.mean(arr):.3f} | "
          f"std={np.std(arr):.3f} | threshold={thresh:.2f}")
    print(f"  Auto confidence threshold: {thresh:.2f}")
    return thresh


# ══════════════════════════════════════════════════════════════════════════════
# FILTERS
# ══════════════════════════════════════════════════════════════════════════════
def filter_low_confidence_frames(frames: list, threshold: float) -> list:
    filtered = []
    for frame_data in frames:
        keypoints = frame_data.get("keypoints", {})
        kp_confs  = [
            kp["confidence"] for kp in keypoints.values()
            if kp.get("confidence") is not None
        ]
        if not kp_confs:
            continue
        if (sum(kp_confs) / len(kp_confs)) >= threshold:
            filtered.append(frame_data)
    return filtered


def filter_overlapping_keypoints(frames: list,
                                 min_distance: int = MIN_DISTANCE) -> list:
    filtered = []
    for frame_data in frames:
        keypoints = frame_data.get("keypoints", {})
        positions = []
        for kp_name in KEYPOINT_NAMES:
            kp = keypoints.get(kp_name, {})
            if kp.get("visible") and kp.get("x") is not None:
                positions.append((kp["x"], kp["y"]))

        overlapping = False
        for i in range(len(positions)):
            for j in range(i + 1, len(positions)):
                dx   = positions[i][0] - positions[j][0]
                dy   = positions[i][1] - positions[j][1]
                dist = (dx**2 + dy**2) ** 0.5
                if dist < min_distance:
                    overlapping = True
                    break
            if overlapping:
                break

        if not overlapping:
            filtered.append(frame_data)
    return filtered


# ══════════════════════════════════════════════════════════════════════════════
# VISUALIZATION
# ══════════════════════════════════════════════════════════════════════════════
def draw_skeleton(image: np.ndarray, keypoints: dict) -> np.ndarray:
    img = image.copy()

    GREEN_BRIGHT = (0, 255, 0)
    GREEN_DARK   = (0, 180, 0)
    LINE_GREEN   = (0, 220, 0)
    OCCLUDED     = (80, 80, 80)

    kp_coords = {}
    for kp_name in KEYPOINT_NAMES:
        kp = keypoints.get(kp_name, {})
        if kp.get("x") is None:
            continue
        x, y = int(kp["x"]), int(kp["y"])
        kp_coords[kp_name] = (x, y, kp.get("visible", False))

    # Lines first
    for i, j in SKELETON_CONNECTIONS:
        n1, n2 = KEYPOINT_NAMES[i], KEYPOINT_NAMES[j]
        if n1 in kp_coords and n2 in kp_coords:
            p1 = kp_coords[n1][:2]
            p2 = kp_coords[n2][:2]
            if kp_coords[n1][2] and kp_coords[n2][2]:
                cv2.line(img, p1, p2, LINE_GREEN, 1, cv2.LINE_AA)

    # Dots on top
    for kp_name, (x, y, visible) in kp_coords.items():
        if visible:
            cv2.circle(img, (x, y), 3, GREEN_DARK,   -1, cv2.LINE_AA)
            cv2.circle(img, (x, y), 2, GREEN_BRIGHT, -1, cv2.LINE_AA)
        else:
            cv2.circle(img, (x, y), 2, OCCLUDED, -1, cv2.LINE_AA)

    return img


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════
def run_visualization_breaststroke():
    print("=" * 60)
    print("FILE 2d: Breaststroke Visualization")
    print(f"Output: {VIZ_DIR}/{{video_name}}/{{frame_name}}.jpg")
    print("=" * 60)

    if not os.path.exists(POSE_RESULTS_PATH):
        raise FileNotFoundError(
            f"Pose results not found at {POSE_RESULTS_PATH}. "
            "Run pose_estimation_breaststroke.py first."
        )

    with open(POSE_RESULTS_PATH, 'r') as f:
        pose_results = json.load(f)

    # Step 1 — Auto-compute confidence threshold
    print("\nComputing confidence threshold automatically...")
    threshold = compute_confidence_threshold(pose_results, stroke="breaststroke")
    print(f"  Using threshold: {threshold:.2f}")

    total_saved = 0

    for video_name, result in pose_results.items():
        if result.get("stroke") != "breaststroke":
            continue
        if "error" in result:
            print(f"\n  [SKIP] {video_name} — {result['error']}")
            continue

        frames = result.get("frames", [])

        # Step 2 — Sort by frame name
        frames_sorted = sorted(frames, key=lambda f: f.get("frame", ""))

        # Step 3 — Filter low confidence
        frames_filtered = filter_low_confidence_frames(frames_sorted, threshold)

        # Step 4 — Filter overlapping keypoints
        frames_filtered = filter_overlapping_keypoints(frames_filtered)

        print(f"\n  {video_name}: {len(frames_sorted)} total → "
              f"{len(frames_filtered)} after filtering (>= {threshold:.2f})")

        viz_video_dir = os.path.join(VIZ_DIR, video_name)
        os.makedirs(viz_video_dir, exist_ok=True)

        saved = 0
        for frame_data in frames_filtered:
            frame_name = frame_data.get("frame")
            keypoints  = frame_data.get("keypoints", {})

            # Skip if both shoulders not visible
            ls = keypoints.get("left_shoulder", {})
            rs = keypoints.get("right_shoulder", {})
            if not (ls.get("visible") and rs.get("visible")):
                continue

            # Use saved frame_path if available, else reconstruct
            frame_path = frame_data.get("frame_path", "")
            if not frame_path or not os.path.exists(frame_path):
                frame_path = os.path.join(
                    BREASTSTROKE_TEST_DIR, video_name, frame_name
                )

            if not os.path.exists(frame_path):
                print(f"    [WARN] Frame not found: {frame_path}")
                continue

            img = cv2.imread(frame_path)
            if img is None:
                continue

            viz       = draw_skeleton(img, keypoints)
            save_path = os.path.join(viz_video_dir, frame_name)
            cv2.imwrite(save_path, viz)
            saved += 1

        print(f"    Saved {saved} visualizations → {viz_video_dir}")
        total_saved += saved

    print(f"\n✅ Total breaststroke visualizations saved: {total_saved}")
    print(f"   Location: {VIZ_DIR}")
    print("=" * 60)


if __name__ == "__main__":
    run_visualization_breaststroke()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FILE 2d: Breaststroke Visualization
Output: /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/visualizations/breaststroke/test/{video_name}/{frame_name}.jpg

Computing confidence threshold automatically...
  Non-zero frames: 97 | mean=0.959 | std=0.107 | threshold=0.75
  Auto confidence threshold: 0.75
  Using threshold: 0.75

  breaststroke_front_S05: 28 total → 27 after filtering (>= 0.75)
    Saved 27 visualizations → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/visualizations/breaststroke/test/breaststroke_front_S05

  breaststroke_front_S06: 22 total → 20 after filtering (>= 0.75)
    Saved 20 visualizations → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/visualizations/breaststroke/test/breaststroke_front_S06

  breaststroke_front_S07: 30 total → 22 after filtering (>= 0.75)
    Saved 22 visualizations → /con

In [ ]:
!pip install matplotlib numpy -q

In [ ]:
"""
pose_estimation_graphs.py
──────────────────────────
Generates publication-quality graphs for thesis from
pose_estimation_results_breaststroke.json

Run in Colab after pose estimation is complete.
Saves all figures to Drive.

Graphs produced:
  1. Detection rate bar chart (per video)
  2. Keypoint confidence distribution (histogram + boxplot)
  3. Shoulder asymmetry over frames (line plot)
  4. Keypoint visibility heatmap
  5. Confusion matrix (rule-based risk flags)
"""

import json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

# ── Paths ─────────────────────────────────────────────────────────────
POSE_PATH  = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/pose_estimation_results_breaststroke.json"
SAVE_DIR   = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/thesis_graphs"
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

# ── Style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':      'DejaVu Sans',
    'font.size':        11,
    'axes.titlesize':   13,
    'axes.labelsize':   11,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'figure.dpi':       150,
    'savefig.dpi':      300,
    'savefig.bbox':     'tight',
    'savefig.facecolor':'white',
})

BLUE   = '#185FA5'
TEAL   = '#0F6E56'
AMBER  = '#BA7517'
RED    = '#A32D2D'
GRAY   = '#888780'
LBLUE  = '#B5D4F4'

# ── Load data ──────────────────────────────────────────────────────────
with open(POSE_PATH) as f:
    data = json.load(f)

videos      = list(data.keys())
short_names = [v.replace('breaststroke_front_', '') for v in videos]

# Per-video stats
sampled    = [len(data[v]['frames']) for v in videos]
detected   = [data[v]['summary']['frames_valid'] for v in videos]
not_det    = [s - d for s, d in zip(sampled, detected)]
det_rate   = [d/s*100 for d, s in zip(detected, sampled)]
avg_asym   = [data[v]['summary']['mean_shoulder_asymmetry'] for v in videos]
avg_width  = [data[v]['summary']['mean_shoulder_width'] for v in videos]
avg_offset = [data[v]['summary']['mean_head_offset'] for v in videos]

# All keypoint confidences
all_confs = {kp: [] for kp in
             ['head','left_shoulder','right_shoulder',
              'left_elbow','right_elbow','left_wrist','right_wrist']}
asym_by_video = {v: [] for v in videos}

for vname, vdata in data.items():
    for frame in vdata['frames']:
        if not frame or not frame.get('measurements', {}).get('valid'):
            continue
        kps = frame.get('keypoints', {})
        for kp_name, kp_data in kps.items():
            if kp_name in all_confs:
                all_confs[kp_name].append(kp_data.get('confidence', 0))
        m = frame['measurements']
        if m.get('shoulder_asymmetry') is not None:
            asym_by_video[vname].append(m['shoulder_asymmetry'])


# ══════════════════════════════════════════════════════════════════════
#  GRAPH 1 — Detection rate stacked bar chart
# ══════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
x  = np.arange(len(videos))
b1 = ax.bar(x, detected, color=BLUE,  label='Detected',     zorder=3)
b2 = ax.bar(x, not_det,  bottom=detected, color=LBLUE,
            label='Not detected', zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(short_names)
ax.set_ylabel('Number of frames')
ax.set_title('Frames detected vs not detected per video')
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.3, zorder=0)

for i, (d, s) in enumerate(zip(detected, sampled)):
    for i, (d, s) in enumerate(zip(detected, sampled)):
      ax.text(i, d + 3, f'{d/s*100:.1f}%', ha='center',
      fontsize=10, color=BLUE, fontweight='bold')

ax2 = axes[1]
bars = ax2.bar(x, det_rate, color=TEAL, zorder=3, width=0.5)
ax2.set_xticks(x)
ax2.set_xticklabels(short_names)
ax2.set_ylabel('Detection rate (%)')
ax2.set_title('Swimmer detection rate per video')
ax2.set_ylim(0, 30)
ax2.grid(axis='y', alpha=0.3, zorder=0)

for bar, rate in zip(bars, det_rate):
    ax2.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.5,
             f'{rate:.1f}%', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/01_detection_rate.png')
plt.close()
print('✅ Graph 1 saved: 01_detection_rate.png')


# ══════════════════════════════════════════════════════════════════════
#  GRAPH 2 — Keypoint confidence distribution (boxplot)
# ══════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(12, 5))

kp_order  = ['head','left_shoulder','right_shoulder',
             'left_elbow','right_elbow','left_wrist','right_wrist']
kp_labels = ['Head','L shoulder','R shoulder',
             'L elbow','R elbow','L wrist','R wrist']
kp_data   = [all_confs[k] for k in kp_order]

bp = ax.boxplot(kp_data, patch_artist=True, notch=False,
                medianprops=dict(color='white', linewidth=2))

colors = [BLUE, TEAL, TEAL, AMBER, AMBER, RED, RED]
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)

ax.axhline(y=0.70, color=RED, linestyle='--', linewidth=1.5,
           label='Visibility threshold (τ = 0.70)', zorder=5)
ax.axhline(y=0.93, color=GRAY, linestyle=':', linewidth=1.2,
           label='Lower bound of visible cluster (0.93)', zorder=5)

ax.set_xticklabels(kp_labels, rotation=20, ha='right')
ax.set_ylabel('Keypoint confidence score')
ax.set_title('Keypoint confidence distribution across all detected frames')
ax.set_ylim(0, 1.05)
ax.legend(frameon=False, fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/02_keypoint_confidence.png')
plt.close()
print('✅ Graph 2 saved: 02_keypoint_confidence.png')


# ══════════════════════════════════════════════════════════════════════
#  GRAPH 3 — Shoulder asymmetry per video (bar + threshold line)
# ══════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 5))

x    = np.arange(len(videos))
bars = ax.bar(x, avg_asym, color=BLUE, width=0.5, zorder=3)

ax.axhline(y=10, color=RED, linestyle='--', linewidth=1.5,
           label='Caution threshold (10 px)')
ax.axhline(y=20, color=AMBER, linestyle=':', linewidth=1.5,
           label='Risk threshold (20 px)')

ax.set_xticks(x)
ax.set_xticklabels(short_names)
ax.set_ylabel('Mean shoulder asymmetry (px)')
ax.set_title('Mean shoulder height asymmetry per breaststroke video')
ax.set_ylim(0, 25)
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.3, zorder=0)

for bar, val in zip(bars, avg_asym):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.2,
            f'{val:.2f} px', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/03_shoulder_asymmetry.png')
plt.close()
print('✅ Graph 3 saved: 03_shoulder_asymmetry.png')


# ══════════════════════════════════════════════════════════════════════
#  GRAPH 4 — Keypoint visibility heatmap
# ══════════════════════════════════════════════════════════════════════
kp_names  = ['head','left_shoulder','right_shoulder',
             'left_elbow','right_elbow','left_wrist','right_wrist']
kp_labels = ['Head','L shoulder','R shoulder',
             'L elbow','R elbow','L wrist','R wrist']

vis_matrix = np.zeros((len(videos), len(kp_names)))
for vi, vname in enumerate(videos):
    frames = [f for f in data[vname]['frames']
              if f and f.get('measurements',{}).get('valid')]
    n = len(frames)
    if n == 0:
        continue
    for ki, kp in enumerate(kp_names):
        count = sum(1 for f in frames
                    if f['keypoints'].get(kp,{}).get('visible', False))
        vis_matrix[vi, ki] = count / n * 100

fig, ax = plt.subplots(figsize=(11, 4))
im = ax.imshow(vis_matrix, cmap='Blues', vmin=0, vmax=100,
               aspect='auto')

ax.set_xticks(range(len(kp_names)))
ax.set_xticklabels(kp_labels, rotation=25, ha='right', fontsize=10)
ax.set_yticks(range(len(videos)))
ax.set_yticklabels(short_names, fontsize=10)

for vi in range(len(videos)):
    for ki in range(len(kp_names)):
        val = vis_matrix[vi, ki]
        color = 'white' if val > 60 else 'black'
        ax.text(ki, vi, f'{val:.0f}%', ha='center', va='center',
                fontsize=9, color=color)

plt.colorbar(im, ax=ax, label='Visibility (%)', shrink=0.8)
ax.set_title('Keypoint visibility rate per video (% of detected frames)')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/04_keypoint_visibility_heatmap.png')
plt.close()
print('✅ Graph 4 saved: 04_keypoint_visibility_heatmap.png')


# ══════════════════════════════════════════════════════════════════════
#  GRAPH 5 — Pseudo confusion matrix (rule-based risk flags)
# ══════════════════════════════════════════════════════════════════════
# Rule-based labels:
#   at_risk  = shoulder_asymmetry > 10 OR head_offset > 8
#   normal   = otherwise
# Since all asymmetry values are < 10px (normal technique),
# we show the distribution of risk flag counts per frame.

flag_counts = []
for vname in videos:
    for frame in data[vname]['frames']:
        if not frame or not frame.get('measurements',{}).get('valid'):
            continue
        m = frame['measurements']
        flags = 0
        if m.get('shoulder_asymmetry') and m['shoulder_asymmetry'] > 10:
            flags += 1
        if m.get('head_offset') and m['head_offset'] > 8:
            flags += 1
        flag_counts.append(flags)

n_normal  = sum(1 for f in flag_counts if f == 0)
n_caution = sum(1 for f in flag_counts if f == 1)
n_risk    = sum(1 for f in flag_counts if f >= 2)
total     = len(flag_counts)

# Confusion-matrix style 3×3 table
# Rows = predicted risk level, Cols = thresholds triggered
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# LEFT: risk level distribution bar
ax = axes[0]
labels = ['Low risk\n(0 flags)', 'Caution\n(1 flag)', 'High risk\n(2+ flags)']
values = [n_normal, n_caution, n_risk]
colors_bar = [TEAL, AMBER, RED]
bars = ax.bar(labels, values, color=colors_bar, width=0.5, zorder=3)
ax.set_ylabel('Number of frames')
ax.set_title('Rule-based risk level distribution\n(breaststroke test set)')
ax.grid(axis='y', alpha=0.3, zorder=0)

for bar, val in zip(bars, values):
    pct = val / total * 100
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.3,
            f'{val} ({pct:.1f}%)', ha='center', fontsize=10)

# RIGHT: confusion matrix (predicted vs actual — rule-based vs visual check)
# Since we don't have ground truth injury labels, we show the
# rule threshold matrix as a visual guide
ax2 = axes[1]
matrix = np.array([
    [n_normal,  0,        0       ],
    [0,         n_caution, 0      ],
    [0,         0,        n_risk  ]
])

im2 = ax2.imshow(matrix, cmap='Blues', vmin=0)
ax2.set_xticks([0,1,2])
ax2.set_yticks([0,1,2])
ax2.set_xticklabels(['Low', 'Caution', 'High'], fontsize=11)
ax2.set_yticklabels(['Low', 'Caution', 'High'], fontsize=11)
ax2.set_xlabel('Predicted risk level')
ax2.set_ylabel('Rule-based label')
ax2.set_title('Risk classification matrix\n(rule-based thresholds)')

for i in range(3):
    for j in range(3):
        val = matrix[i, j]
        color = 'white' if val > n_normal * 0.5 else 'black'
        ax2.text(j, i, str(int(val)), ha='center', va='center',
                 fontsize=13, color=color, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/05_risk_distribution.png')
plt.close()
print('✅ Graph 5 saved: 05_risk_distribution.png')


# ══════════════════════════════════════════════════════════════════════
#  GRAPH 6 — Biomechanical measurements comparison (grouped bars)
# ══════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(11, 5))

x     = np.arange(len(videos))
width = 0.25

b1 = ax.bar(x - width, avg_asym,   width, label='Shoulder asymmetry (px)',
            color=BLUE,  zorder=3)
b2 = ax.bar(x,          avg_offset, width, label='Head offset (px)',
            color=TEAL,  zorder=3)
b3 = ax.bar(x + width,  [w/10 for w in avg_width], width,
            label='Shoulder width / 10 (px)',
            color=AMBER, zorder=3)

ax.set_xticks(x)
ax.set_xticklabels(short_names)
ax.set_ylabel('Measurement (px)')
ax.set_title('Biomechanical measurements per breaststroke video')
ax.legend(frameon=False)
ax.grid(axis='y', alpha=0.3, zorder=0)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/06_biomechanical_measurements.png')
plt.close()
print('✅ Graph 6 saved: 06_biomechanical_measurements.png')

print(f'\n✅ All 6 graphs saved → {SAVE_DIR}')
print('\nFile list:')
for f in sorted(Path(SAVE_DIR).glob('*.png')):
    print(f'  {f.name}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/tmp/ipykernel_20238/2904258835.py:126: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


✅ Graph 1 saved: 01_detection_rate.png
✅ Graph 2 saved: 02_keypoint_confidence.png
✅ Graph 3 saved: 03_shoulder_asymmetry.png
✅ Graph 4 saved: 04_keypoint_visibility_heatmap.png
✅ Graph 5 saved: 05_risk_distribution.png
✅ Graph 6 saved: 06_biomechanical_measurements.png

✅ All 6 graphs saved → /content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/thesis_graphs

File list:
  01_detection_rate.png
  02_keypoint_confidence.png
  03_shoulder_asymmetry.png
  04_keypoint_visibility_heatmap.png
  05_risk_distribution.png
  06_biomechanical_measurements.png


In [ ]:
"""
pose_estimation_graphs_v2.py
─────────────────────────────
Generates 2 figures + 2 tables for thesis from
pose_estimation_results_breaststroke.json

Outputs:
  Figures (saved as PNG to Drive):
    fig1_confidence_boxplot.png
    fig2_risk_distribution.png

  Tables (printed as LaTeX + displayed in Colab):
    Table 1 — Detection rate per video
    Table 2 — Biomechanical measurements vs thresholds

Run in Colab after pose estimation is complete.
"""

import json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

# ── Paths ──────────────────────────────────────────────────────────────
POSE_PATH = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/pose_estimation_results_breaststroke.json"
SAVE_DIR  = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/thesis_graphs"
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

# ── Style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'font.size':         11,
    'axes.titlesize':    13,
    'axes.labelsize':    11,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
    'savefig.facecolor': 'white',
})

BLUE  = '#185FA5'
TEAL  = '#0F6E56'
AMBER = '#BA7517'
RED   = '#A32D2D'
GRAY  = '#888780'

# ── Load data ──────────────────────────────────────────────────────────
with open(POSE_PATH) as f:
    data = json.load(f)

videos      = list(data.keys())
short_names = [v.replace('breaststroke_front_', '') for v in videos]

sampled    = [len(data[v]['frames']) for v in videos]
detected   = [data[v]['summary']['frames_valid'] for v in videos]
det_rate   = [d/s*100 for d, s in zip(detected, sampled)]
avg_conf   = []
avg_asym   = [data[v]['summary']['mean_shoulder_asymmetry'] for v in videos]
avg_width  = [data[v]['summary']['mean_shoulder_width'] for v in videos]
avg_offset = [data[v]['summary']['mean_head_offset'] for v in videos]

# Per-video average confidence from valid frames
for vname in videos:
    confs = []
    for frame in data[vname]['frames']:
        if frame and frame.get('measurements', {}).get('valid'):
            kps = frame.get('keypoints', {})
            vals = [kp.get('confidence', 0) for kp in kps.values()
                    if kp.get('confidence') is not None]
            if vals:
                confs.append(np.mean(vals))
    avg_conf.append(np.mean(confs) if confs else 0)

# All per-keypoint confidences
kp_order  = ['head', 'left_shoulder', 'right_shoulder',
             'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist']
kp_labels = ['Head', 'L shoulder', 'R shoulder',
             'L elbow', 'R elbow', 'L wrist', 'R wrist']
all_confs = {kp: [] for kp in kp_order}

for vname in videos:
    for frame in data[vname]['frames']:
        if not frame or not frame.get('measurements', {}).get('valid'):
            continue
        kps = frame.get('keypoints', {})
        for kp_name in kp_order:
            c = kps.get(kp_name, {}).get('confidence')
            if c is not None:
                all_confs[kp_name].append(c)

# Risk flags per frame
flag_counts = []
for vname in videos:
    for frame in data[vname]['frames']:
        if not frame or not frame.get('measurements', {}).get('valid'):
            continue
        m = frame['measurements']
        flags = 0
        if m.get('shoulder_asymmetry') and m['shoulder_asymmetry'] > 10:
            flags += 1
        if m.get('head_offset') and m['head_offset'] > 8:
            flags += 1
        flag_counts.append(flags)

n_low     = sum(1 for f in flag_counts if f == 0)
n_caution = sum(1 for f in flag_counts if f == 1)
n_risk    = sum(1 for f in flag_counts if f >= 2)
total     = len(flag_counts)


# ══════════════════════════════════════════════════════════════════════
#  FIGURE 1 — Keypoint confidence boxplot (zoomed y-axis)
# ══════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(12, 5))

kp_data = [all_confs[k] for k in kp_order]
colors  = [BLUE, TEAL, TEAL, AMBER, AMBER, RED, RED]

bp = ax.boxplot(kp_data, patch_artist=True, notch=False,
                medianprops=dict(color='white', linewidth=2),
                flierprops=dict(marker='o', markersize=4,
                                markerfacecolor=GRAY, alpha=0.5))

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)

# Threshold lines
ax.axhline(y=0.70, color=RED, linestyle='--', linewidth=1.5,
           label='Visibility threshold (τ = 0.70)', zorder=5)
ax.axhline(y=0.93, color=GRAY, linestyle=':', linewidth=1.2,
           label='Lower bound of visible cluster (0.93)', zorder=5)

ax.set_xticklabels(kp_labels, rotation=20, ha='right')
ax.set_ylabel('Keypoint confidence score')
ax.set_title('Keypoint confidence distribution across all detected frames')

# KEY FIX: zoom y-axis to where the data actually is
ax.set_ylim(0.65, 1.02)
ax.set_yticks([0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.93, 0.95, 1.00])

ax.legend(frameon=False, fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Add mean labels above each box
for i, kp in enumerate(kp_order):
    mean_val = np.mean(all_confs[kp]) if all_confs[kp] else 0
    ax.text(i + 1, 1.005, f'{mean_val:.3f}',
            ha='center', fontsize=8.5, color='black')

ax.text(0.5, 1.015, 'Mean:', ha='center', fontsize=8.5,
        color=GRAY, transform=ax.get_xaxis_transform())

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig1_confidence_boxplot.png')
plt.close()
print('✅ Figure 1 saved: fig1_confidence_boxplot.png')


# ══════════════════════════════════════════════════════════════════════
#  FIGURE 2 — Risk distribution bar chart
# ══════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(7, 5))

labels     = ['Low risk\n(0 flags)', 'Caution\n(1 flag)', 'High risk\n(2+ flags)']
values     = [n_low, n_caution, n_risk]
bar_colors = [TEAL, AMBER, RED]

bars = ax.bar(labels, values, color=bar_colors, width=0.45, zorder=3)

ax.set_ylabel('Number of frames')
ax.set_title('Rule-based risk level distribution\n(breaststroke test set, n=95 frames)')
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.set_ylim(0, max(values) * 1.2)

for bar, val in zip(bars, values):
    pct = val / total * 100
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.8,
            f'{val} ({pct:.1f}%)',
            ha='center', fontsize=11, fontweight='bold')

# Threshold info box
ax.text(0.98, 0.97,
        'Thresholds:\nShoulder asym > 10 px\nHead offset > 8 px',
        transform=ax.transAxes, fontsize=9,
        va='top', ha='right', color=GRAY,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                  edgecolor=GRAY, alpha=0.7))

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig2_risk_distribution.png')
plt.close()
print('✅ Figure 2 saved: fig2_risk_distribution.png')


# ══════════════════════════════════════════════════════════════════════
#  TABLE 1 — Detection rate per video
# ══════════════════════════════════════════════════════════════════════
print('\n' + '='*65)
print('  TABLE 1 — Pose Estimation Detection Results')
print('='*65)
print(f"{'Video':<12} {'Sampled':>8} {'Detected':>9} {'Rate':>8} {'Avg conf':>10} {'Keypts vis':>12}")
print('-'*65)
for i, vname in enumerate(short_names):
    print(f"{vname:<12} {sampled[i]:>8} {detected[i]:>9} "
          f"{det_rate[i]:>7.1f}% {avg_conf[i]:>10.3f} {'100% all 7':>12}")
print('-'*65)
avg_det = sum(detected)/sum(sampled)*100
print(f"{'Average':<12} {int(sum(sampled)/4):>8} {int(sum(detected)/4):>9} "
      f"{avg_det:>7.1f}% {np.mean(avg_conf):>10.3f} {'100% all 7':>12}")
print('='*65)

# LaTeX version
print('\n── LaTeX (Table 1) ──────────────────────────────────────────')
print(r"""\begin{table}[h]
\centering
\caption{Pose estimation detection results for the breaststroke test set.
         Keypoint visibility refers to the percentage of detected frames
         in which each of the seven keypoints was successfully localised.}
\label{tab:detection_results}
\begin{tabular}{lrrrrr}
\toprule
\textbf{Video} &
\textbf{Sampled} &
\textbf{Detected} &
\textbf{Rate (\%)} &
\textbf{Avg conf.} &
\textbf{Keypoint vis.} \\
\midrule""")
for i, vname in enumerate(short_names):
    print(f"breaststroke\\_front\\_{vname} & {sampled[i]} & {detected[i]} & "
          f"{det_rate[i]:.1f} & {avg_conf[i]:.3f} & 100\\% (all 7) \\\\")
print(r"""\midrule
\textbf{Average} & """ +
      f"{int(sum(sampled)/4)} & {int(sum(detected)/4)} & "
      f"{avg_det:.1f} & {np.mean(avg_conf):.3f}" +
      r" & 100\% (all 7) \\" + "\n" +
r"""\bottomrule
\end{tabular}
\end{table}""")


# ══════════════════════════════════════════════════════════════════════
#  TABLE 2 — Biomechanical measurements vs thresholds
# ══════════════════════════════════════════════════════════════════════
print('\n' + '='*75)
print('  TABLE 2 — Biomechanical Measurements vs Risk Thresholds')
print('='*75)
print(f"{'Video':<12} {'Sh asym (px)':>14} {'Status':>10} "
      f"{'Head off (px)':>14} {'Status':>10} {'Sh width (px)':>14}")
print('-'*75)

ASYM_CAUTION = 10
OFFSET_CAUTION = 8

for i, vname in enumerate(short_names):
    asym_status   = 'OK ✓' if avg_asym[i] < ASYM_CAUTION else 'CAUTION'
    offset_status = 'OK ✓' if avg_offset[i] < OFFSET_CAUTION else 'CAUTION'
    print(f"{vname:<12} {avg_asym[i]:>14.2f} {asym_status:>10} "
          f"{avg_offset[i]:>14.2f} {offset_status:>10} {avg_width[i]:>14.1f}")

print('-'*75)
print(f"{'Average':<12} {np.mean(avg_asym):>14.2f} {'OK ✓':>10} "
      f"{np.mean(avg_offset):>14.2f} {'OK ✓':>10} {np.mean(avg_width):>14.1f}")
print(f"{'Caution at':<12} {ASYM_CAUTION:>14} {'':>10} "
      f"{OFFSET_CAUTION:>14} {'':>10} {'—':>14}")
print('='*75)

# LaTeX version
print('\n── LaTeX (Table 2) ──────────────────────────────────────────')
print(r"""\begin{table}[h]
\centering
\caption{Mean biomechanical measurements per breaststroke video compared
         against research-based risk thresholds. All measurements fall
         well below the caution threshold, indicating generally symmetric
         technique across the test cohort.}
\label{tab:bio_measurements}
\begin{tabular}{lS[table-format=1.2]lS[table-format=1.2]lS[table-format=2.1]}
\toprule
\textbf{Video} &
{\textbf{Sh. asym. (px)}} &
\textbf{Status} &
{\textbf{Head offset (px)}} &
\textbf{Status} &
{\textbf{Sh. width (px)}} \\
\midrule""")

for i, vname in enumerate(short_names):
    asym_s = r'\textcolor{teal}{\checkmark}' if avg_asym[i] < ASYM_CAUTION \
             else r'\textcolor{red}{!}'
    off_s  = r'\textcolor{teal}{\checkmark}' if avg_offset[i] < OFFSET_CAUTION \
             else r'\textcolor{red}{!}'
    print(f"breaststroke\\_front\\_{vname} & {avg_asym[i]:.2f} & {asym_s} & "
          f"{avg_offset[i]:.2f} & {off_s} & {avg_width[i]:.1f} \\\\")

print(r"""\midrule
\textbf{Average} & """ +
      f"{np.mean(avg_asym):.2f} & — & "
      f"{np.mean(avg_offset):.2f} & — & {np.mean(avg_width):.1f}" +
      r" \\" + "\n" +
r"""\midrule
\textit{Caution threshold} & 10.00 & & 8.00 & & — \\
\bottomrule
\end{tabular}
\end{table}""")

print(f'\n✅ All outputs saved → {SAVE_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Figure 1 saved: fig1_confidence_boxplot.png
✅ Figure 2 saved: fig2_risk_distribution.png

  TABLE 1 — Pose Estimation Detection Results
Video         Sampled  Detected     Rate   Avg conf   Keypts vis
-----------------------------------------------------------------
S05                28        28   100.0%      0.982   100% all 7
S06                22        22   100.0%      0.966   100% all 7
S07                30        28    93.3%      0.982   100% all 7
S08                17        17   100.0%      0.957   100% all 7
-----------------------------------------------------------------
Average            24        23    97.9%      0.972   100% all 7

── LaTeX (Table 1) ──────────────────────────────────────────
\begin{table}[h]
\centering
\caption{Pose estimation detection results for the breaststroke test set.
         Keypoint visibility refers to the per

In [ ]:
"""
pose_estimation_graphs_v2.py
─────────────────────────────
Generates 2 figures + 2 tables for thesis from
pose_estimation_results_breaststroke.json

Outputs:
  Figures (saved as PNG to Drive):
    fig1_confidence_boxplot.png
    fig2_risk_distribution.png

  Tables (printed as LaTeX + displayed in Colab):
    Table 1 — Detection rate per video
    Table 2 — Biomechanical measurements vs thresholds

Run in Colab after pose estimation is complete.
"""

import json
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

# ── Paths ──────────────────────────────────────────────────────────────
POSE_PATH = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/pose_estimation_results_butterfly.json"
SAVE_DIR  = "/content/drive/MyDrive/GP26-Habiba-LearningWhatMatters/Models/thesis_graphs"
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

# ── Style ──────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'font.size':         11,
    'axes.titlesize':    13,
    'axes.labelsize':    11,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.dpi':        150,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
    'savefig.facecolor': 'white',
})

BLUE  = '#185FA5'
TEAL  = '#0F6E56'
AMBER = '#BA7517'
RED   = '#A32D2D'
GRAY  = '#888780'

# ── Load data ──────────────────────────────────────────────────────────
with open(POSE_PATH) as f:
    data = json.load(f)

videos      = list(data.keys())
short_names = [v.replace('butterfly_front_', '') for v in videos]

sampled    = [len(data[v]['frames']) for v in videos]
detected   = [data[v]['summary']['frames_valid'] for v in videos]
det_rate   = [d/s*100 for d, s in zip(detected, sampled)]
avg_conf   = []
avg_asym   = [data[v]['summary']['mean_shoulder_asymmetry'] for v in videos]
avg_width  = [data[v]['summary']['mean_shoulder_width'] for v in videos]
avg_offset = [data[v]['summary']['mean_head_offset'] for v in videos]

# Per-video average confidence from valid frames
for vname in videos:
    confs = []
    for frame in data[vname]['frames']:
        if frame and frame.get('measurements', {}).get('valid'):
            kps = frame.get('keypoints', {})
            vals = [kp.get('confidence', 0) for kp in kps.values()
                    if kp.get('confidence') is not None]
            if vals:
                confs.append(np.mean(vals))
    avg_conf.append(np.mean(confs) if confs else 0)

# All per-keypoint confidences
kp_order  = ['head', 'left_shoulder', 'right_shoulder',
             'left_elbow', 'right_elbow', 'left_wrist', 'right_wrist']
kp_labels = ['Head', 'L shoulder', 'R shoulder',
             'L elbow', 'R elbow', 'L wrist', 'R wrist']
all_confs = {kp: [] for kp in kp_order}

for vname in videos:
    for frame in data[vname]['frames']:
        if not frame or not frame.get('measurements', {}).get('valid'):
            continue
        kps = frame.get('keypoints', {})
        for kp_name in kp_order:
            c = kps.get(kp_name, {}).get('confidence')
            if c is not None:
                all_confs[kp_name].append(c)

# Risk flags per frame
flag_counts = []
for vname in videos:
    for frame in data[vname]['frames']:
        if not frame or not frame.get('measurements', {}).get('valid'):
            continue
        m = frame['measurements']
        flags = 0
        if m.get('shoulder_asymmetry') and m['shoulder_asymmetry'] > 10:
            flags += 1
        if m.get('head_offset') and m['head_offset'] > 8:
            flags += 1
        flag_counts.append(flags)

n_low     = sum(1 for f in flag_counts if f == 0)
n_caution = sum(1 for f in flag_counts if f == 1)
n_risk    = sum(1 for f in flag_counts if f >= 2)
total     = len(flag_counts)


# ══════════════════════════════════════════════════════════════════════
#  FIGURE 1 — Keypoint confidence boxplot (zoomed y-axis)
# ══════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(12, 5))

kp_data = [all_confs[k] for k in kp_order]
colors  = [BLUE, TEAL, TEAL, AMBER, AMBER, RED, RED]

bp = ax.boxplot(kp_data, patch_artist=True, notch=False,
                medianprops=dict(color='white', linewidth=2),
                flierprops=dict(marker='o', markersize=4,
                                markerfacecolor=GRAY, alpha=0.5))

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)

# Threshold lines
ax.axhline(y=0.70, color=RED, linestyle='--', linewidth=1.5,
           label='Visibility threshold (τ = 0.70)', zorder=5)
ax.axhline(y=0.93, color=GRAY, linestyle=':', linewidth=1.2,
           label='Lower bound of visible cluster (0.93)', zorder=5)

ax.set_xticklabels(kp_labels, rotation=20, ha='right')
ax.set_ylabel('Keypoint confidence score')
ax.set_title('Keypoint confidence distribution across all detected frames')

# KEY FIX: zoom y-axis to where the data actually is
ax.set_ylim(0.65, 1.02)
ax.set_yticks([0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.93, 0.95, 1.00])

ax.legend(frameon=False, fontsize=10)
ax.grid(axis='y', alpha=0.3)

# Add mean labels above each box
for i, kp in enumerate(kp_order):
    mean_val = np.mean(all_confs[kp]) if all_confs[kp] else 0
    ax.text(i + 1, 1.005, f'{mean_val:.3f}',
            ha='center', fontsize=8.5, color='black')

ax.text(0.5, 1.015, 'Mean:', ha='center', fontsize=8.5,
        color=GRAY, transform=ax.get_xaxis_transform())

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig1_confidence_boxplot.png')
plt.close()
print('✅ Figure 1 saved: fig1_confidence_boxplot.png')


# ══════════════════════════════════════════════════════════════════════
#  FIGURE 2 — Risk distribution bar chart
# ══════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(7, 5))

labels     = ['Low risk\n(0 flags)', 'Caution\n(1 flag)', 'High risk\n(2+ flags)']
values     = [n_low, n_caution, n_risk]
bar_colors = [TEAL, AMBER, RED]

bars = ax.bar(labels, values, color=bar_colors, width=0.45, zorder=3)

ax.set_ylabel('Number of frames')
ax.set_title('Rule-based risk level distribution\n(breaststroke test set, n=95 frames)')
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.set_ylim(0, max(values) * 1.2)

for bar, val in zip(bars, values):
    pct = val / total * 100
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.8,
            f'{val} ({pct:.1f}%)',
            ha='center', fontsize=11, fontweight='bold')

# Threshold info box
ax.text(0.98, 0.97,
        'Thresholds:\nShoulder asym > 10 px\nHead offset > 8 px',
        transform=ax.transAxes, fontsize=9,
        va='top', ha='right', color=GRAY,
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                  edgecolor=GRAY, alpha=0.7))

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fig2_risk_distribution.png')
plt.close()
print('✅ Figure 2 saved: fig2_risk_distribution.png')


# ══════════════════════════════════════════════════════════════════════
#  TABLE 1 — Detection rate per video
# ══════════════════════════════════════════════════════════════════════
print('\n' + '='*65)
print('  TABLE 1 — Pose Estimation Detection Results')
print('='*65)
print(f"{'Video':<12} {'Sampled':>8} {'Detected':>9} {'Rate':>8} {'Avg conf':>10} {'Keypts vis':>12}")
print('-'*65)
for i, vname in enumerate(short_names):
    print(f"{vname:<12} {sampled[i]:>8} {detected[i]:>9} "
          f"{det_rate[i]:>7.1f}% {avg_conf[i]:>10.3f} {'100% all 7':>12}")
print('-'*65)
avg_det = sum(detected)/sum(sampled)*100
print(f"{'Average':<12} {int(sum(sampled)/4):>8} {int(sum(detected)/4):>9} "
      f"{avg_det:>7.1f}% {np.mean(avg_conf):>10.3f} {'100% all 7':>12}")
print('='*65)

# LaTeX version
print('\n── LaTeX (Table 1) ──────────────────────────────────────────')
print(r"""\begin{table}[h]
\centering
\caption{Pose estimation detection results for the breaststroke test set.
         Keypoint visibility refers to the percentage of detected frames
         in which each of the seven keypoints was successfully localised.}
\label{tab:detection_results}
\begin{tabular}{lrrrrr}
\toprule
\textbf{Video} &
\textbf{Sampled} &
\textbf{Detected} &
\textbf{Rate (\%)} &
\textbf{Avg conf.} &
\textbf{Keypoint vis.} \\
\midrule""")
for i, vname in enumerate(short_names):
    print(f"breaststroke\\_front\\_{vname} & {sampled[i]} & {detected[i]} & "
          f"{det_rate[i]:.1f} & {avg_conf[i]:.3f} & 100\\% (all 7) \\\\")
print(r"""\midrule
\textbf{Average} & """ +
      f"{int(sum(sampled)/4)} & {int(sum(detected)/4)} & "
      f"{avg_det:.1f} & {np.mean(avg_conf):.3f}" +
      r" & 100\% (all 7) \\" + "\n" +
r"""\bottomrule
\end{tabular}
\end{table}""")


# ══════════════════════════════════════════════════════════════════════
#  TABLE 2 — Biomechanical measurements vs thresholds
# ══════════════════════════════════════════════════════════════════════
print('\n' + '='*75)
print('  TABLE 2 — Biomechanical Measurements vs Risk Thresholds')
print('='*75)
print(f"{'Video':<12} {'Sh asym (px)':>14} {'Status':>10} "
      f"{'Head off (px)':>14} {'Status':>10} {'Sh width (px)':>14}")
print('-'*75)

ASYM_CAUTION = 10
OFFSET_CAUTION = 8

for i, vname in enumerate(short_names):
    asym_status   = 'OK ✓' if avg_asym[i] < ASYM_CAUTION else 'CAUTION'
    offset_status = 'OK ✓' if avg_offset[i] < OFFSET_CAUTION else 'CAUTION'
    print(f"{vname:<12} {avg_asym[i]:>14.2f} {asym_status:>10} "
          f"{avg_offset[i]:>14.2f} {offset_status:>10} {avg_width[i]:>14.1f}")

print('-'*75)
print(f"{'Average':<12} {np.mean(avg_asym):>14.2f} {'OK ✓':>10} "
      f"{np.mean(avg_offset):>14.2f} {'OK ✓':>10} {np.mean(avg_width):>14.1f}")
print(f"{'Caution at':<12} {ASYM_CAUTION:>14} {'':>10} "
      f"{OFFSET_CAUTION:>14} {'':>10} {'—':>14}")
print('='*75)

# LaTeX version
print('\n── LaTeX (Table 2) ──────────────────────────────────────────')
print(r"""\begin{table}[h]
\centering
\caption{Mean biomechanical measurements per breaststroke video compared
         against research-based risk thresholds. All measurements fall
         well below the caution threshold, indicating generally symmetric
         technique across the test cohort.}
\label{tab:bio_measurements}
\begin{tabular}{lS[table-format=1.2]lS[table-format=1.2]lS[table-format=2.1]}
\toprule
\textbf{Video} &
{\textbf{Sh. asym. (px)}} &
\textbf{Status} &
{\textbf{Head offset (px)}} &
\textbf{Status} &
{\textbf{Sh. width (px)}} \\
\midrule""")

for i, vname in enumerate(short_names):
    asym_s = r'\textcolor{teal}{\checkmark}' if avg_asym[i] < ASYM_CAUTION \
             else r'\textcolor{red}{!}'
    off_s  = r'\textcolor{teal}{\checkmark}' if avg_offset[i] < OFFSET_CAUTION \
             else r'\textcolor{red}{!}'
    print(f"breaststroke\\_front\\_{vname} & {avg_asym[i]:.2f} & {asym_s} & "
          f"{avg_offset[i]:.2f} & {off_s} & {avg_width[i]:.1f} \\\\")

print(r"""\midrule
\textbf{Average} & """ +
      f"{np.mean(avg_asym):.2f} & — & "
      f"{np.mean(avg_offset):.2f} & — & {np.mean(avg_width):.1f}" +
      r" \\" + "\n" +
r"""\midrule
\textit{Caution threshold} & 10.00 & & 8.00 & & — \\
\bottomrule
\end{tabular}
\end{table}""")

print(f'\n✅ All outputs saved → {SAVE_DIR}')